# Agent Loop: AI Engineering Skill Coach

This final project combines **function calling** with the **Gradio chat interface** from the previous module. The model may answer directly, request one of our Python functions, or request more functions after seeing a result.

The app runs each requested function and keeps asking the model until it produces an answer. That repeated decision-and-tool cycle is the **agent loop**.

In [1]:
import json

import gradio as gr
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=True)

client = OpenAI()
model = "gpt-6-luna"

/Users/lukaslechner/PythonProjects/AI-Engineering-Foundations-Labs/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Reuse our first function

This is the same `in_demand_ai_skills()` function from `2-function-calling.ipynb`. The percentages are **illustrative lesson data**, not current job-market statistics.

In [2]:
def in_demand_ai_skills():
    return [
        {"skill": "python", "percentage": 88.1},
        {"skill": "llm", "percentage": 57.1},
        {"skill": "rag", "percentage": 45.2},
        {"skill": "aws", "percentage": 40.5},
        {"skill": "prompt engineering", "percentage": 35.7},
        {"skill": "langchain", "percentage": 28.6},
        {"skill": "azure", "percentage": 23.8},
        {"skill": "machine learning", "percentage": 19.0},
        {"skill": "generative ai", "percentage": 19.0},
        {"skill": "gcp", "percentage": 19.0},
        {"skill": "ai agents", "percentage": 19.0},
        {"skill": "tensorflow", "percentage": 16.7},
        {"skill": "fine-tuning", "percentage": 16.7},
        {"skill": "pytorch", "percentage": 16.7},
        {"skill": "vector databases", "percentage": 14.3},
        {"skill": "nlp", "percentage": 14.3},
        {"skill": "docker", "percentage": 14.3},
        {"skill": "typescript", "percentage": 14.3},
        {"skill": "langgraph", "percentage": 14.3},
        {"skill": "sql", "percentage": 11.9},
        {"skill": "kubernetes", "percentage": 11.9},
        {"skill": "llamaindex", "percentage": 11.9},
        {"skill": "cicd", "percentage": 11.9},
        {"skill": "mcp", "percentage": 11.9},
        {"skill": "embeddings", "percentage": 11.9},
    ]

## 2. Add a function with an argument

The second function compares a learner's existing skills with the example list. It returns the five highest-ranked skills they have not listed yet. It does not assess proficiency or predict hiring outcomes.

In [3]:
def recommend_skills_to_learn(known_skills):
    known = {skill.strip().lower() for skill in known_skills}
    missing_skills = [
        skill for skill in in_demand_ai_skills() if skill["skill"] not in known
    ]
    return missing_skills[:5]


recommend_skills_to_learn(["Python", "RAG"])

[{'skill': 'llm', 'percentage': 57.1},
 {'skill': 'aws', 'percentage': 40.5},
 {'skill': 'prompt engineering', 'percentage': 35.7},
 {'skill': 'langchain', 'percentage': 28.6},
 {'skill': 'azure', 'percentage': 23.8}]

## 3. Describe both functions to the model

The model sees these schemas, not our Python code. `strict=True` makes the argument shape predictable. The model can choose either function, both functions, or neither.

In [ ]:
tools = [
    {
        "type": "function",
        "name": "in_demand_ai_skills",
        "description": "Get the 25 most in-demand AI engineering skills with illustrative percentages. Use for questions about the example skill rankings.",
        "parameters": {
            "type": "object",
            "properties": {},
            "required": [],
            "additionalProperties": False,
        },
        "strict": True,
    },
    {
        "type": "function",
        "name": "recommend_skills_to_learn",
        "description": "Given skills a learner already knows, return the five highest-ranked missing skills from the illustrative example data. Use for personalized study suggestions.",
        "parameters": {
            "type": "object",
            "properties": {
                "known_skills": {
                    "type": "array",
                    "items": {"type": "string"},
                    "description": "AI engineering skills the learner says they already know",
                }
            },
            "required": ["known_skills"],
            "additionalProperties": False,
        },
        "strict": True,
    },
]

## 4. Build the loop

Gradio supplies the visible chat history. We send that history and the new user message to the Responses API. After every response, we keep **all** output items, including reasoning and function-call items, before adding function results. Each result uses the matching `call_id`.

If there are no function calls, the response is ready for the chat UI. A five-round limit prevents a mistaken tool cycle from running forever.

In [5]:
instructions = (
    "You are an AI engineering study coach. "
    "The skill percentages are illustrative lesson data, not current job-market statistics. "
    "Use the functions when the user asks about the example skill data or what to study next. "
    "For unrelated questions, answer directly."
)


def chat(message, history):
    conversation = [
        {
            "role": item["role"],
            "content": item["content"][0]["text"],
        }
        for item in history
    ]
    conversation.append({"role": "user", "content": message})

    for _ in range(5):
        response = client.responses.create(
            model=model,
            instructions=instructions,
            input=conversation,
            tools=tools,
            tool_choice="auto",
        )
        conversation.extend(response.output)
        tool_calls = [item for item in response.output if item.type == "function_call"]

        if not tool_calls:
            return response.output_text

        for tool_call in tool_calls:
            print(f"Calling {tool_call.name} with {tool_call.arguments}")
            try:
                arguments = json.loads(tool_call.arguments)
                if tool_call.name == "in_demand_ai_skills":
                    result = in_demand_ai_skills()
                elif tool_call.name == "recommend_skills_to_learn":
                    result = recommend_skills_to_learn(arguments["known_skills"])
                else:
                    result = {"error": f"Unknown function: {tool_call.name}"}
            except (KeyError, TypeError, ValueError) as error:
                result = {"error": str(error)}

            conversation.append(
                {
                    "type": "function_call_output",
                    "call_id": tool_call.call_id,
                    "output": json.dumps(result),
                }
            )

    return "I reached the tool-call limit. Please try a simpler question."

The `print` line shows each requested function in the notebook output. It helps you see when the model used a tool and when it answered on its own.

The visible Gradio history contains user messages and final assistant answers. Tool-call details are kept **within the current turn** so the model can finish that turn.

## 5. Try the chat

Try these prompts in order:

1. “Hi! What can you help me with?” — likely a direct answer.
2. “Which cloud skills appear in your example data?” — likely a call to `in_demand_ai_skills`.
3. “I know Python and RAG. What should I study next?” — likely a call to `recommend_skills_to_learn`.
4. “Why did you suggest the first one?” — tests the conversation history.

The model chooses whether to call a tool, so its choice can vary.

In [6]:
demo = gr.ChatInterface(
    fn=chat,
    title="AI Engineering Skill Coach",
    description="Explore illustrative skill data and get study suggestions.",
)
demo.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


Calling in_demand_ai_skills with {}
Calling recommend_skills_to_learn with {"known_skills":["python","RAG"]}
Calling recommend_skills_to_learn with {"known_skills":["Python","RAG"]}


## Your turn

Add one more question to the chat that needs the model to use both functions, or change the recommendation function to return a different number of skills. Watch the printed tool requests and explain why the loop needed one or more model requests.

**Common mistake:** Returning a function's Python result directly skips the model's final answer. The result belongs in a `function_call_output` item, followed by another model request.

**Optional extension:** Add a third function that looks up one named skill in the same example data.

### Sources

- [OpenAI function calling guide](https://developers.openai.com/api/docs/guides/function-calling)
- [Gradio 6 chat history format](https://gradio.app/guides/gradio-6-migration-guide)